# 02. ML 모델 비교 분석

**목적**: Ridge, XGBoost, CatBoost 모델의 성능을 비교하고,
Rule-based 방식으로 전환할 수밖에 없었던 논리적 근거를 제시합니다.

## 1. 데이터 로드

In [19]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# 데이터 로드 (portfolio 폴더 기준 상대경로)
df = pd.read_csv("../dataset/merged_dataset.csv")
print(f"데이터 shape: {df.shape}")
df.head(10)

데이터 shape: (175, 25)


,region,year,single_house_total,apartment_total,row_house_total,multi_house_total,non_residential_housing_total,target_value,aging_index,population_total,...,under_20,age_65_over,age_0_14,cpi_index,low_income_elderly_65_79_ratio,low_income_elderly_80_over_ratio,basic_pension_recipient_count,basic_pension_recipient_ratio,alone_household_count,elderly_population
0,강남구,2017,8487,7204,432,4553,1027,8,11.09,522514,...,539,7607,68591,97.645,4.64,6.81,1706,22.43,11396.0,8502.0
1,강남구,2018,8593,6715,433,4935,1057,12,11.75,507810,...,421,7834,66657,99.086,45.66,36.18,1840,23.49,11749.0,8819.0
2,강남구,2019,8876,7115,450,5286,1087,10,12.57,509199,...,607,8393,66767,99.466,1.68,4.44,2959,35.26,12358.0,9490.0
3,강남구,2020,8859,7413,453,5716,1026,11,13.90,508135,...,498,9162,65904,100.000,8.87,7.07,3994,43.59,13592.0,10335.0
4,강남구,2021,8861,8410,516,6038,1053,12,15.95,503019,...,397,10292,64520,102.500,2.19,5.48,3970,38.57,15432.0,11593.0
5,강남구,2022,8776,8925,541,6263,1008,13,17.71,500149,...,465,11031,62278,107.720,1.17,2.49,3492,31.66,19102.0,12442.0
6,강남구,2023,8821,9838,575,6533,1012,11,18.56,511084,...,342,11860,63911,111.590,2.04,3.91,3660,30.86,20973.0,13459.0
7,강동구,2017,9735,3300,143,3244,751,19,14.46,423978,...,162,7506,51894,97.645,5.59,15.33,2415,32.17,12055.0,8207.0
8,강동구,2018,9655,3235,152,3608,755,27,16.04,414231,...,144,7917,49358,99.086,7.10,12.66,2304,29.10,12524.0,8666.0
9,강동구,2019,9839,3860,131,4100,767,24,17.45,415287,...,157,8694,49823,99.466,6.86,25.22,2891,33.25,13191.0,9523.0


## 2. 데이터 전처리

In [20]:
# 고독사율 계산
df = df[df["elderly_population"] > 0].copy()
df["death_rate"] = df["target_value"] / df["elderly_population"]

# 노인 인구 비율
df["elderly_population_ratio"] = df["age_65_over"] / df["population_total"] * 100

# 피처 정의
feature_cols = [
    "single_household_ratio",
    "low_income_elderly_65_79_ratio",
    "low_income_elderly_80_over_ratio",
    "aging_index",
    "elderly_population_ratio",
]

# 결측치 처리
df = df.dropna(subset=feature_cols + ["death_rate"])
print(f"전처리 후 데이터 수: {len(df)}개")
df.head(10)

전처리 후 데이터 수: 168개


,region,year,single_house_total,apartment_total,row_house_total,multi_house_total,non_residential_housing_total,target_value,aging_index,population_total,...,age_0_14,cpi_index,low_income_elderly_65_79_ratio,low_income_elderly_80_over_ratio,basic_pension_recipient_count,basic_pension_recipient_ratio,alone_household_count,elderly_population,death_rate,elderly_population_ratio
0,강남구,2017,8487,7204,432,4553,1027,8,11.09,522514,...,68591,97.645,4.64,6.81,1706,22.43,11396.0,8502.0,0.000941,1.455846
1,강남구,2018,8593,6715,433,4935,1057,12,11.75,507810,...,66657,99.086,45.66,36.18,1840,23.49,11749.0,8819.0,0.001361,1.542703
2,강남구,2019,8876,7115,450,5286,1087,10,12.57,509199,...,66767,99.466,1.68,4.44,2959,35.26,12358.0,9490.0,0.001054,1.648275
3,강남구,2020,8859,7413,453,5716,1026,11,13.90,508135,...,65904,100.000,8.87,7.07,3994,43.59,13592.0,10335.0,0.001064,1.803064
4,강남구,2021,8861,8410,516,6038,1053,12,15.95,503019,...,64520,102.500,2.19,5.48,3970,38.57,15432.0,11593.0,0.001035,2.046046
5,강남구,2022,8776,8925,541,6263,1008,13,17.71,500149,...,62278,107.720,1.17,2.49,3492,31.66,19102.0,12442.0,0.001045,2.205543
6,강남구,2023,8821,9838,575,6533,1012,11,18.56,511084,...,63911,111.590,2.04,3.91,3660,30.86,20973.0,13459.0,0.000817,2.320558
7,강동구,2017,9735,3300,143,3244,751,19,14.46,423978,...,51894,97.645,5.59,15.33,2415,32.17,12055.0,8207.0,0.002315,1.770375
8,강동구,2018,9655,3235,152,3608,755,27,16.04,414231,...,49358,99.086,7.10,12.66,2304,29.10,12524.0,8666.0,0.003116,1.911252
9,강동구,2019,9839,3860,131,4100,767,24,17.45,415287,...,49823,99.466,6.86,25.22,2891,33.25,13191.0,9523.0,0.002520,2.093492


## 3. 데이터 분할 (시계열 기반)

In [12]:
# 시계열 분할: 2017-2022 학습, 2023 테스트
TRAIN_START = 2017
TRAIN_END = 2022
TEST_YEAR = 2023

train_df = df[(df["year"] >= TRAIN_START) & (df["year"] <= TRAIN_END)]
test_df = df[df["year"] == TEST_YEAR]

X_train = train_df[feature_cols]
y_train = train_df["death_rate"]
X_test = test_df[feature_cols]
y_test = test_df["death_rate"]

print(f"학습 데이터: {len(X_train)}개 ({TRAIN_START}~{TRAIN_END})")
print(f"테스트 데이터: {len(X_test)}개 ({TEST_YEAR})")

학습 데이터: 144개 (2017~2022)
테스트 데이터: 24개 (2023)


## 4. 모델 비교

### 4.1 Linear Regression

In [13]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print("[Linear Regression]")
print(f"  RMSE: {rmse_lr:.6f}")
print(f"  R² : {r2_lr:.4f}")
print(f"  MAE : {mae_lr:.6f}")

[Linear Regression]
  RMSE: 0.001174
  R² : -1.9036
  MAE : 0.001036


### 4.2 Ridge Regression (with GridSearchCV)

In [14]:
tscv = TimeSeriesSplit(n_splits=3)
alpha_grid = np.logspace(-4, 2, 20)

ridge = Ridge()
grid_ridge = GridSearchCV(
    ridge,
    {"alpha": alpha_grid},
    scoring="neg_mean_squared_error",
    cv=tscv,
    n_jobs=-1
)
grid_ridge.fit(X_train, y_train)

print(f"최적 alpha: {grid_ridge.best_params_['alpha']:.6f}")
print(f"CV MSE: {-grid_ridge.best_score_:.8f}")

y_pred_ridge = grid_ridge.predict(X_test)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
r2_ridge = r2_score(y_test, y_pred_ridge)

print(f"\n[Ridge Regression]")
print(f"  RMSE: {rmse_ridge:.6f}")
print(f"  R² : {r2_ridge:.4f}")

최적 alpha: 0.000100
CV MSE: 0.00000095

[Ridge Regression]
  RMSE: 0.001174
  R² : -1.9036


### 4.3 XGBoost (with GridSearchCV)

In [15]:
param_grid_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [2, 3],
    "learning_rate": [0.03, 0.1],
}

xgb = XGBRegressor(objective="reg:squarederror", random_state=42)
grid_xgb = GridSearchCV(
    xgb,
    param_grid_xgb,
    scoring="neg_mean_squared_error",
    cv=tscv,
    n_jobs=-1
)
grid_xgb.fit(X_train, y_train)

print(f"최적 파라미터: {grid_xgb.best_params_}")
print(f"CV MSE: {-grid_xgb.best_score_:.8f}")

y_pred_xgb = grid_xgb.predict(X_test)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"\n[XGBoost]")
print(f"  RMSE: {rmse_xgb:.6f}")
print(f"  R² : {r2_xgb:.4f}")

최적 파라미터: {'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 200}
CV MSE: 0.00000110

[XGBoost]
  RMSE: 0.001035
  R² : -1.2573


### 4.4 CatBoost (with GridSearchCV)

In [16]:
param_grid_cat = {
    "depth": [3, 4, 5],
    "learning_rate": [0.03, 0.1],
    "iterations": [200, 400],
}

cat = CatBoostRegressor(loss_function="RMSE", random_seed=42, verbose=False)
grid_cat = GridSearchCV(
    cat,
    param_grid_cat,
    scoring="neg_mean_squared_error",
    cv=tscv,
    n_jobs=-1
)
grid_cat.fit(X_train, y_train)

print(f"최적 파라미터: {grid_cat.best_params_}")
print(f"CV MSE: {-grid_cat.best_score_:.8f}")

y_pred_cat = grid_cat.predict(X_test)
rmse_cat = np.sqrt(mean_squared_error(y_test, y_pred_cat))
r2_cat = r2_score(y_test, y_pred_cat)

print(f"\n[CatBoost]")
print(f"  RMSE: {rmse_cat:.6f}")
print(f"  R² : {r2_cat:.4f}")

최적 파라미터: {'depth': 5, 'iterations': 400, 'learning_rate': 0.03}
CV MSE: 0.00000115

[CatBoost]
  RMSE: 0.001115
  R² : -1.6181


## 5. 모델 성능 비교 요약

In [17]:
results = pd.DataFrame({
    "Model": ["LinearRegression", "Ridge", "XGBoost", "CatBoost"],
    "RMSE": [rmse_lr, rmse_ridge, rmse_xgb, rmse_cat],
    "R²": [r2_lr, r2_ridge, r2_xgb, r2_cat],
})

print("\n" + "="*50)
print("모델 성능 비교")
print("="*50)
print(results.to_string(index=False))
print("="*50)


모델 성능 비교
           Model     RMSE        R²
LinearRegression 0.001174 -1.903593
           Ridge 0.001174 -1.903585
         XGBoost 0.001035 -1.257260
        CatBoost 0.001115 -1.618077


## 6. 결론: Rule-based로 전환한 이유

### 6.1 문제점 분석

1. **소규모 데이터**: 175개 샘플 (25개 구 × 7년)
   - ML 모델이 일반화하기에 부족한 데이터
   - 과적합 위험 높음

2. **낮은 R² 값**
   - 대부분의 모델이 0.3 이하의 R²
   - 설명력이 매우 부족

3. **시계열 특성 미반영**
   - 연도별 트렌드를 충분히 학습하지 못함
   - TimeSeriesSplit 사용해도 개선 미미

### 6.2 Rule-based 방식의 장점

1. **해석 가능성**: 상관계수 기반 가중치로 명확한 설명 가능
2. **안정성**: 소규모 데이터에서도 일관된 결과
3. **도메인 지식 활용**: 전문가 검토 가능
4. **구별 가중치 조정**: 지역 특성 반영 가능

In [18]:
# global_corr.ipynb에서 계산된 상관계수 기반 가중치
global_weights = {
    "single_household_ratio": 0.368467,
    "low_income_elderly_65_79_ratio": 0.038977,
    "low_income_elderly_80_over_ratio": 0.095469,
    "aging_index": 0.311710,
    "elderly_population_ratio": 0.185376,
}

print("\n상관계수 기반 전역 가중치 (Rule-based 기반):")
for feat, weight in sorted(global_weights.items(), key=lambda x: -x[1]):
    print(f"  {feat}: {weight:.4f}")


상관계수 기반 전역 가중치 (Rule-based 기반):
  single_household_ratio: 0.3685
  aging_index: 0.3117
  elderly_population_ratio: 0.1854
  low_income_elderly_80_over_ratio: 0.0955
  low_income_elderly_65_79_ratio: 0.0390
